In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor
import logging
import numpy as np
import h5py
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from flax import nnx
from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from heavyhex_qft.triangular_z2 import TriangularZ2Lattice
from heavyhex_qft.plaquette_dual import PlaquetteDual
from skqd_z2lgt.circuits import make_step_circuits, compose_trotter_circuits
from skqd_z2lgt.recovery_learning import preprocess
from skqd_z2lgt.crbm import ConditionalRBM
from skqd_z2lgt.train_crbm import DefaultCallback, NLLCallback, cd_percloss, nll_loss, train_crbm

os.environ['CUDA_VISIBLE_DEVICES'] = '8'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.99'
jax.config.update('jax_enable_x64', True)

logging.basicConfig(level=logging.WARNING)
logging.getLogger('skqd_z2lgt').setLevel(logging.INFO)
LOG = logging.getLogger()
LOG.setLevel(logging.INFO)

In [ ]:
logging.getLogger('jax').setLevel(logging.WARNING)

In [ ]:
output_file = h5py.File('/data/iiyama/2dz2/recovery_learning.h5', 'r+')

In [ ]:
lattice = TriangularZ2Lattice('''
 * * *
* * * *
 * * *
''')
plaquette_energy = 0.8
delta_t = 0.5
num_steps = 5
basis_2q = 'cz'

dual_lattice = PlaquetteDual(lattice)
ising_hamiltonian = dual_lattice.make_hamiltonian(plaquette_energy)

In [ ]:
lattice.draw_graph();

## Compose the circuits and run the experiments

In [ ]:
def run_experiments(lattice, backend):
    layout = lattice.layout_heavy_hex(backend.coupling_map, backend_properties=backend.properties(), basis_2q=basis_2q)
    full_step, fwd_step, bkd_step, measure = transpile(
        make_step_circuits(lattice, plaquette_energy, delta_t, basis_2q),
        backend=backend, initial_layout=layout, optimization_level=2
    )
    id_step = fwd_step.compose(bkd_step)
    exp_circuits = compose_trotter_circuits(full_step, measure, num_steps)
    ref_circuits = compose_trotter_circuits(id_step, measure, num_steps)

    sampler = Sampler(backend)
    job = sampler.run(exp_circuits + ref_circuits, shots=backend.configuration().max_shots)
    return job

## Convert link-state counts to plaquette-state arrays with MWPM

In [ ]:
try:
    data = output_file['data'][()]
except KeyError:
    compute = True
else:
    compute = False

if compute:
    try:
        del output_file['data']
    except KeyError:
        pass

    service = QiskitRuntimeService(instance='ICEPP-dedicated-temp-prem-us')

    try:
        job_id = output_file['job_id'][()].decode()
    except KeyError:
        backend = service.backend('ibm_pittsburgh')
        job = run_experiments(lattice, backend)
        output_file.create_dataset('job_id', data=job.job_id())
    else:
        job = service.job(job_id)

    result = job.result()

    with ProcessPoolExecutor() as executor:
        futures = []
        for res in result[5:]:
            futures.append(executor.submit(preprocess, res.data.c.get_counts(), dual_lattice))
        step_plaq_data = [future.result() for future in futures]

    data = np.array(step_plaq_data)
    output_file.create_dataset('data', data=data)

In [ ]:
num_vertices = lattice.num_vertices
num_plaquettes = lattice.num_plaquettes

In [ ]:
istep = 0

num_h = 256
num_epochs = 50
batch_size = 32
loss = 'nll'
lr = 0.001

model_filename = f'/data/iiyama/2dz2/crbm_models/p10_i{istep}_h{num_h}_b{batch_size}_lr{lr:.0e}_e{num_epochs}_{loss}_w1.h5'

train_dataset = data[istep][:80_000]
test_dataset = data[istep][80_000:]

compute = True

if not compute and os.path.exists(model_filename):
    model = ConditionalRBM.load(model_filename)
    with h5py.File(model_filename.replace('.h5', '_records.h5'), 'r') as source:
        group = source['records']
        records = {key: array[()] for key, array in group.items()}
else:
    mean_activation = np.mean(train_dataset[:, num_vertices:], axis=0)
    mean_activation = np.where(np.isclose(mean_activation, 0.), 1.e-6, mean_activation)
    bias_init = np.log(mean_activation / (1. - mean_activation))
    h_sparsity = 0.001

    rngs = nnx.Rngs(params=0, sample=1)
    model = ConditionalRBM(num_vertices, num_plaquettes, num_h, rngs=rngs)
    model.bias_v.value = bias_init
    model.bias_h.value = jnp.full(model.bias_h.shape, np.log(h_sparsity / (1. - h_sparsity)))

    if loss == 'cd':
        loss_fn = cd_percloss(l2_weight=1.)
        callback = DefaultCallback(eval_every=100)
    elif loss == 'nll':
        loss_fn = nll_loss(l2_weight=1.)
        callback = NLLCallback(eval_every=100)

    records = train_crbm(model, train_dataset, test_dataset, batch_size, num_epochs, loss_fn, lr,
                         callback=callback)
    model.save(model_filename)
    with h5py.File(model_filename.replace('.h5', '_records.h5'), 'w') as out:
        group = out.create_group('records')
        for key, array in records.items():
            group.create_dataset(key, data=array)

In [ ]:
evals_per_epoch = 80000 // batch_size // 100
x_train = np.linspace(0., num_epochs, evals_per_epoch * num_epochs)
x_test = np.arange(0, num_epochs) + 1
plt.plot(x_train, records['train_loss'], label='train')
plt.plot(x_test, records['test_loss'], label='test')
plt.legend();

In [ ]:
plt.plot(x_train, records['train_free_energy'], label='train')
plt.plot(x_test, records['test_free_energy'], label='test')
plt.legend();

In [ ]:
u_states, inverse, u_counts = np.unique(train_dataset[:, :num_vertices], return_counts=True, return_inverse=True, axis=0)
u_states = u_states[u_counts > 500]
u_indices = np.nonzero(u_counts > 500)[0]
v_states_gen = []
for u_state in u_states:
    v_states_gen.append(model.sample(u_state, size=10000))

In [ ]:
fig, axs = plt.subplots(5, 4, sharex=True, sharey=True)
for uid, u_state, v_gen, ax in zip(u_indices, u_states, v_states_gen, fig.axes):
    mask = inverse == uid
    v_data = train_dataset[mask, num_vertices:]
    ax.bar(np.arange(num_plaquettes), np.mean(v_gen, axis=0), width=0.9, label='gen')
    ax.bar(np.arange(num_plaquettes), np.mean(v_data, axis=0), width=0.5, label='data')

In [ ]:
fig, axs = plt.subplots(5, 4, sharex=True, sharey=True)
for uid, u_state, v_gen, ax in zip(u_indices, u_states, v_states_gen, fig.axes):
    mask = inverse == uid
    v_data = train_dataset[mask, num_vertices:]
    ax.hist(np.sum(v_data, axis=1), np.linspace(-0.5, 10.5, 12))
    counts, bins = np.histogram(np.sum(v_gen, axis=1), np.linspace(-0.5, 10.5, 12))
    ax.stairs(counts / v_gen.shape[0] * v_data.shape[0], bins)

In [ ]:
counts, bins = np.histogram(np.array(model.weights_hv.value).reshape(-1))
plt.stairs(counts / np.prod(model.weights_hv.shape), bins, label='weights_hv')
counts, bins = np.histogram(np.array(model.bias_h.value).reshape(-1))
plt.stairs(counts / np.prod(model.bias_h.shape), bins, label='bias_h')
counts, bins = np.histogram(np.array(model.bias_v.value).reshape(-1))
plt.stairs(counts / np.prod(model.bias_v.shape), bins, label='bias_v')
plt.legend();